# Preparacion del conjunto de datos (Silver -> Gold con Spark)

Objetivo: preparar el dataset etiquetado de mood para entrenamiento.

Alcance: limpieza, consistencia, tratamiento de outliers, features derivadas, escalado y split.

Fuente unica: capa Silver en S3. Procesamiento completo con Apache Spark.

Salida: dataset preparado en S3 (Gold).

In [11]:
import os
import sys
import shutil
from pathlib import Path
from typing import Dict, List, Tuple

import boto3
import numpy as np
import pandas as pd
from pyspark.sql import functions as F

PROJECT_ROOT = Path.cwd().parent
sys.path.append(str(PROJECT_ROOT))
from src.config import load_settings
from src.spark.session import build_spark

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 120)

settings = load_settings()
spark = build_spark("music-recommender-data-prep")

# Temporary local folder for Silver data downloaded from S3
local_silver_base = settings.project_root / "data_lake" / "tmp_silver"
local_silver_mood = local_silver_base / "mood_dataset"
local_silver_tracks = local_silver_base / "tracks_dataset"
local_silver_mood.mkdir(parents=True, exist_ok=True)
local_silver_tracks.mkdir(parents=True, exist_ok=True)

s3 = boto3.client("s3", region_name=settings.aws_region)

def download_prefix(bucket: str, prefix: str, local_dir: Path) -> None:
    if local_dir.exists():
        shutil.rmtree(local_dir)
    local_dir.mkdir(parents=True, exist_ok=True)
    paginator = s3.get_paginator("list_objects_v2")
    for page in paginator.paginate(Bucket=bucket, Prefix=prefix):
        for obj in page.get("Contents", []):
            key = obj["Key"]
            if key.endswith("/"):
                continue
            rel = key[len(prefix):]
            if not rel:
                continue
            target = local_dir / rel
            target.parent.mkdir(parents=True, exist_ok=True)
            s3.download_file(bucket, key, str(target))

settings

Settings(project_root=WindowsPath('C:/Users/losal/Desktop/CursoEspecializacion/TFG'), data_dir=WindowsPath('C:/Users/losal/Desktop/CursoEspecializacion/TFG/datasets'), local_lake_dir=WindowsPath('C:/Users/losal/Desktop/CursoEspecializacion/TFG/data_lake'), bucket_name='music-recommender-data-lake-807744154206', aws_region='us-east-1', s3_endpoint_url=None, use_s3=True, kafka_bootstrap_servers='localhost:9092', kafka_group_id='music-recommender-pipeline', max_rows_per_dataset=200, producer_batch_size=25, producer_batch_delay_seconds=0.5, consumer_timeout_seconds=20, mongo_uri='mongodb://localhost:27017', mongo_database='music_recommender', rds_dsn='dbname=music_recommender user=postgres password=postgres host=localhost port=5432')

In [12]:
silver_prefix_mood = "silver/mood_dataset/"
silver_prefix_tracks = "silver/tracks_dataset/"
download_prefix(settings.bucket_name, silver_prefix_mood, local_silver_mood)
download_prefix(settings.bucket_name, silver_prefix_tracks, local_silver_tracks)

mood_silver = spark.read.parquet(str(local_silver_mood))
tracks_silver = spark.read.parquet(str(local_silver_tracks))

print("Silver mood:", f"s3://{settings.bucket_name}/{silver_prefix_mood}")
print("Silver tracks:", f"s3://{settings.bucket_name}/{silver_prefix_tracks}")
print("Local temp (mood):", local_silver_mood)
print("Local temp (tracks):", local_silver_tracks)
print("Rows mood:", mood_silver.count())
print("Rows tracks:", tracks_silver.count())

Silver mood: s3://music-recommender-data-lake-807744154206/silver/mood_dataset/
Silver tracks: s3://music-recommender-data-lake-807744154206/silver/tracks_dataset/
Local temp (mood): C:\Users\losal\Desktop\CursoEspecializacion\TFG\data_lake\tmp_silver\mood_dataset
Local temp (tracks): C:\Users\losal\Desktop\CursoEspecializacion\TFG\data_lake\tmp_silver\tracks_dataset
Rows mood: 5000
Rows tracks: 3980


## 1. Comprension general (mood)
Se valida estructura, tipos y nulos del dataset etiquetado.

In [13]:
def overview_spark(df, name: str, preview_rows: int = 3):
    print(name, "rows", df.count())
    display(df.limit(preview_rows).toPandas())
    df.printSchema()
    null_exprs = [F.mean(F.col(c).isNull().cast("double")).alias(c) for c in df.columns]
    nulls = df.select(*null_exprs).toPandas().T.rename(columns={0: "null_pct"})
    display(nulls.sort_values("null_pct", ascending=False).head(10))

overview_spark(mood_silver, "mood_silver")
overview_spark(tracks_silver, "tracks_silver")

mood_silver rows 5000


,acousticness,danceability,duration_ms,energy,instrumentalness,mood_label,liveness,loudness,spec_rate,speechiness,tempo,uri,valence
0,0.0467,0.639,163870,0.788,0.0,1,0.0738,-6.658,6.956734e-07,0.1140,154.085,spotify:track:6tAaye3AnUXe2ODwl4KtfL,0.541
1,0.0278,0.681,156319,0.711,0.0,1,0.3120,-4.790,5.412010e-07,0.0846,89.890,spotify:track:1PAvhqefivTNdxZ4u8RT1d,0.826
2,0.4550,0.620,174987,0.808,0.0,1,0.0758,-6.566,2.183019e-07,0.0382,150.768,spotify:track:6rpljHv0NRZ5pH8gV2zG4U,0.788


root
 |-- acousticness: double (nullable = true)
 |-- danceability: double (nullable = true)
 |-- duration_ms: long (nullable = true)
 |-- energy: double (nullable = true)
 |-- instrumentalness: double (nullable = true)
 |-- mood_label: integer (nullable = true)
 |-- liveness: double (nullable = true)
 |-- loudness: double (nullable = true)
 |-- spec_rate: double (nullable = true)
 |-- speechiness: double (nullable = true)
 |-- tempo: double (nullable = true)
 |-- uri: string (nullable = true)
 |-- valence: double (nullable = true)



,null_pct
acousticness,0.0
danceability,0.0
duration_ms,0.0
energy,0.0
instrumentalness,0.0
mood_label,0.0
liveness,0.0
loudness,0.0
spec_rate,0.0
speechiness,0.0


tracks_silver rows 3980


,acousticness,album_name,artists,danceability,duration_ms,energy,explicit,instrumentalness,key,liveness,loudness,mode,popularity,speechiness,tempo,time_signature,track_genre,track_id,track_name,valence
0,0.918,What You Need,Chord Overstreet,0.519,181786,0.264,False,0.000253,0,0.237,-11.171,1,53,0.0601,140.354,4,acoustic,00DuOxungwHwVUX2m37P0P,What You Need,0.192
1,0.289,Days I Will Remember,Tyrone Wells,0.688,214240,0.481,False,0.000000,6,0.189,-8.807,1,58,0.1050,98.017,4,acoustic,01MVOl9KtVTNfFiBU9I7dc,Days I Will Remember,0.666
2,0.789,Bossa Best,Suy Galvez,0.710,181466,0.646,False,0.000004,3,0.106,-5.945,1,28,0.0313,120.028,4,acoustic,02EqvposhJjwrgkQexKIdW,Isn't She Lovely,0.615


root
 |-- acousticness: double (nullable = true)
 |-- album_name: string (nullable = true)
 |-- artists: string (nullable = true)
 |-- danceability: double (nullable = true)
 |-- duration_ms: integer (nullable = true)
 |-- energy: double (nullable = true)
 |-- explicit: boolean (nullable = true)
 |-- instrumentalness: double (nullable = true)
 |-- key: integer (nullable = true)
 |-- liveness: double (nullable = true)
 |-- loudness: double (nullable = true)
 |-- mode: integer (nullable = true)
 |-- popularity: integer (nullable = true)
 |-- speechiness: double (nullable = true)
 |-- tempo: double (nullable = true)
 |-- time_signature: integer (nullable = true)
 |-- track_genre: string (nullable = true)
 |-- track_id: string (nullable = true)
 |-- track_name: string (nullable = true)
 |-- valence: double (nullable = true)



,null_pct
acousticness,0.0
album_name,0.0
artists,0.0
danceability,0.0
duration_ms,0.0
energy,0.0
explicit,0.0
instrumentalness,0.0
key,0.0
liveness,0.0


## 2. Seleccion de variables base
Mood: se eliminan columnas no utiles (como `uri`).
Tracks: se conserva `track_id` y solo las columnas compartidas con mood para asegurar compatibilidad.

In [14]:
MOOD_FEATURES = [
    "danceability", "energy", "speechiness", "acousticness",
    "instrumentalness", "liveness", "valence", "loudness", "tempo", "spec_rate", "duration_ms"
 ]
TRACKS_FEATURES = [
    "danceability", "energy", "speechiness", "acousticness",
    "instrumentalness", "liveness", "valence", "loudness", "tempo", "duration_ms"
 ]

mood_features = [c for c in MOOD_FEATURES if c in mood_silver.columns]
tracks_features = [c for c in TRACKS_FEATURES if c in tracks_silver.columns]
shared_features = [c for c in mood_features if c in tracks_features]
if "mood_label" not in mood_silver.columns:
    raise ValueError("mood_label no existe en Silver")
if "track_id" not in tracks_silver.columns:
    raise ValueError("track_id no existe en Silver")

mood_base = mood_silver.select(["mood_label", *shared_features])
tracks_base = tracks_silver.select(["track_id", *shared_features])

# Normalizar tipos numericos y eliminar columnas no necesarias (uri)
for col in shared_features:
    mood_base = mood_base.withColumn(col, F.col(col).cast("double"))
    tracks_base = tracks_base.withColumn(col, F.col(col).cast("double"))
mood_base = mood_base.withColumn("mood_label", F.col("mood_label").cast("int"))

tracks_base

DataFrame[track_id: string, danceability: double, energy: double, speechiness: double, acousticness: double, instrumentalness: double, liveness: double, valence: double, loudness: double, tempo: double, duration_ms: double]

## 3. Limpieza: nulos e inconsistencias
Se corrigen tipos y se imputan nulos con medianas (sin eliminar mas filas de lo necesario).

In [15]:
def impute_with_median(df, numeric_cols: List[str]):
    medians = {}
    for col in numeric_cols:
        quantiles = df.approxQuantile(col, [0.5], 0.001)
        medians[col] = quantiles[0] if quantiles else None
    for col, median in medians.items():
        if median is not None:
            df = df.withColumn(col, F.coalesce(F.col(col), F.lit(median)))
    return df

# Mood: eliminar filas sin target
mood_clean = mood_base.filter(F.col("mood_label").isNotNull())
mood_clean = impute_with_median(mood_clean, shared_features)

# Tracks: eliminar filas sin track_id y duplicados por track_id
tracks_clean = tracks_base.filter(F.col("track_id").isNotNull()).dropDuplicates(["track_id"])
tracks_clean = impute_with_median(tracks_clean, shared_features)

print("Rows mood after cleaning:", mood_clean.count())
print("Rows tracks after cleaning:", tracks_clean.count())

Rows mood after cleaning: 5000
Rows tracks after cleaning: 1000


## 4. Outliers
Se aplica clipping IQR por feature para reducir extremos sin borrar registros.

In [16]:
def clip_outliers_iqr(df, cols: List[str]):
    for col in cols:
        q1 = df.approxQuantile(col, [0.25], 0.001)[0]
        q3 = df.approxQuantile(col, [0.75], 0.001)[0]
        iqr = q3 - q1
        lower = q1 - 1.5 * iqr
        upper = q3 + 1.5 * iqr
        df = df.withColumn(col, F.when(F.col(col) < lower, lower).when(F.col(col) > upper, upper).otherwise(F.col(col)))
    return df

mood_clipped = clip_outliers_iqr(mood_clean, shared_features)
tracks_clipped = clip_outliers_iqr(tracks_clean, shared_features)

mood_clipped

DataFrame[mood_label: int, danceability: double, energy: double, speechiness: double, acousticness: double, instrumentalness: double, liveness: double, valence: double, loudness: double, tempo: double, duration_ms: double]

## 5. Ingenieria de features
Se mantiene el conjunto de columnas original para asegurar compatibilidad con el dataset de clasificacion.

In [17]:
# No se crean columnas nuevas para mantener compatibilidad con el dataset de clasificacion
mood_fe = mood_clipped
tracks_fe = tracks_clipped

mood_fe

DataFrame[mood_label: int, danceability: double, energy: double, speechiness: double, acousticness: double, instrumentalness: double, liveness: double, valence: double, loudness: double, tempo: double, duration_ms: double]

## 6. Escalado
Se estandarizan variables numericas con z-score usando estadisticas del dataset.

In [18]:
feature_cols = [c for c in mood_fe.columns if c != "mood_label"]

# Compute mean/std for scaling (from mood)
stats_rows = []
stats_map = {}
for col in feature_cols:
    row = mood_fe.agg(
        F.mean(F.col(col)).alias("mean"),
        F.stddev(F.col(col)).alias("std"),
    ).collect()[0]
    mean_val = row["mean"]
    std_val = row["std"] or 1.0
    stats_rows.append({"feature": col, "mean": float(mean_val), "std": float(std_val)})
    stats_map[col] = (mean_val, std_val)
    mood_fe = mood_fe.withColumn(col, (F.col(col) - F.lit(mean_val)) / F.lit(std_val))

mood_scaled = mood_fe
scaler_stats = pd.DataFrame(stats_rows)
scaler_stats.head()


,feature,mean,std
0,danceability,0.623034,0.151098
1,energy,0.683577,0.184910
2,speechiness,0.061941,0.037793
3,acousticness,0.207053,0.230595
4,instrumentalness,0.003286,0.005370


In [19]:
# Apply the same scaling to tracks (only shared features)
tracks_scaled = tracks_fe
for col, (mean_val, std_val) in stats_map.items():
    if col in tracks_scaled.columns:
        tracks_scaled = tracks_scaled.withColumn(col, (F.col(col) - F.lit(mean_val)) / F.lit(std_val))
tracks_scaled

DataFrame[track_id: string, danceability: double, energy: double, speechiness: double, acousticness: double, instrumentalness: double, liveness: double, valence: double, loudness: double, tempo: double, duration_ms: double]

## 7. Split train/test
Split estratificado 80/20 por `mood_label`.

In [20]:
labels = [row[0] for row in mood_scaled.select("mood_label").distinct().collect()]
train_parts = []
test_parts = []
for label in labels:
    subset = mood_scaled.filter(F.col("mood_label") == label)
    train_part, test_part = subset.randomSplit([0.8, 0.2], seed=42)
    train_parts.append(train_part)
    test_parts.append(test_part)

train_df = train_parts[0]
for part in train_parts[1:]:
    train_df = train_df.unionByName(part)
test_df = test_parts[0]
for part in test_parts[1:]:
    test_df = test_df.unionByName(part)

print("Train rows:", train_df.count(), "Test rows:", test_df.count())
train_df.groupBy("mood_label").count().show()
test_df.groupBy("mood_label").count().show()

Train rows: 4017 Test rows: 983
+----------+-----+
|mood_label|count|
+----------+-----+
|         1| 2235|
|         3|   26|
|         2|  772|
|         0|  984|
+----------+-----+

+----------+-----+
|mood_label|count|
+----------+-----+
|         1|  555|
|         3|    4|
|         2|  193|
|         0|  231|
+----------+-----+



## 8. Guardado en Gold (S3)
Se escriben train/test y estadisticas de escalado en S3 Gold.

In [21]:
gold_local = settings.project_root / "data_lake" / "tmp_gold" / "mood_prepared"
tracks_local = settings.project_root / "data_lake" / "tmp_gold" / "tracks_prepared"
for path in [gold_local, tracks_local]:
    if path.exists():
        shutil.rmtree(path)
    path.mkdir(parents=True, exist_ok=True)

train_local = gold_local / "train"
test_local = gold_local / "test"
scaler_local = gold_local / "scaler_stats"
tracks_out_local = tracks_local / "full"

train_df.write.mode("overwrite").parquet(str(train_local))
test_df.write.mode("overwrite").parquet(str(test_local))
spark.createDataFrame(scaler_stats).write.mode("overwrite").parquet(str(scaler_local))
tracks_scaled.write.mode("overwrite").parquet(str(tracks_out_local))

def upload_directory(local_dir: Path, bucket: str, prefix: str) -> None:
    for path in local_dir.rglob("*"):
        if path.is_file():
            rel = path.relative_to(local_dir).as_posix()
            key = f"{prefix}/{rel}"
            s3.upload_file(str(path), bucket, key)

upload_directory(gold_local, settings.bucket_name, "gold/mood_prepared")
upload_directory(tracks_local, settings.bucket_name, "gold/tracks_prepared")

print("Saved and uploaded to:")
print("-", f"s3://{settings.bucket_name}/gold/mood_prepared/train")
print("-", f"s3://{settings.bucket_name}/gold/mood_prepared/test")
print("-", f"s3://{settings.bucket_name}/gold/mood_prepared/scaler_stats")
print("-", f"s3://{settings.bucket_name}/gold/tracks_prepared/full")

Saved and uploaded to:
- s3://music-recommender-data-lake-807744154206/gold/mood_prepared/train
- s3://music-recommender-data-lake-807744154206/gold/mood_prepared/test
- s3://music-recommender-data-lake-807744154206/gold/mood_prepared/scaler_stats
- s3://music-recommender-data-lake-807744154206/gold/tracks_prepared/full
